In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
teste = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/test.csv")
teste.shape

In [ ]:
treino = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/train.csv")
treino.shape

In [ ]:
coluna = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/trainLabels.csv")
coluna.head()

In [ ]:
#treino.info()
teste.info()

In [ ]:
#treino.isnull().sum()
teste.isnull().sum()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(25, 15))
correlation = treino.select_dtypes('number').corr()
sns.heatmap(correlation, ax=ax, linewidths=0.5, cmap='mako_r', annot=True)

In [ ]:
import numpy as np
data_nova = pd.concat([treino, coluna], axis = 1)
data_nova.columns = np.arange(41)

In [ ]:
data_nova

In [ ]:
# Função para identificar os outliers de uma coluna
def identify_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return column[(column < lower_bound) | (column > upper_bound)]

# Lista para armazenar os nomes das colunas com outliers
colunas_com_outliers = []

# Iterar sobre cada coluna do DataFrame
for coluna in data_nova.columns:
    outliers = identify_outliers(data_nova[coluna])
    if not outliers.empty:
        colunas_com_outliers.append(coluna)
        data_sem_outliers = data_nova[~data_nova[coluna].isin(outliers)]

# Mostrar os nomes das colunas com outliers
print("Colunas com outliers:", colunas_com_outliers)


In [ ]:
data_sem_outliers

In [ ]:
X_final = data_sem_outliers.drop([40], axis = 1)
y_final = data_sem_outliers[40]

In [ ]:
from sklearn.model_selection import train_test_split
X_treino, X_teste, y_treino, y_teste = train_test_split(X_final, y_final, 
                                                       test_size = 0.7, 
                                                       random_state = 42)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

def modelo_funcao (X_treino, X_teste, y_treino, y_teste, modelo, nome_modelo):
    modelo.fit(X_treino, y_treino)
    previsao = modelo.predict(X_teste)
    resultado_treino = modelo.score(X_treino, y_treino)
    resultado_teste = modelo.score(X_teste, y_teste)
    resultado = accuracy_score(y_teste, previsao)
    print(f'Nome do modelo:       {nome_modelo}', f'Resultado treino:   {resultado_treino}', f'Resultado teste:        {resultado_teste}', f'Resultado Final:   {resultado}', sep='\n')

In [ ]:
LR = LogisticRegression()
modelo_funcao(X_treino, X_teste, y_treino, y_teste, LR, "Regressao Logistica")

In [ ]:
submission = pd.DataFrame(LR.predict(teste))
print(submission.shape)
submission.columns = ['Solution']
submission['Id'] = np.arange(1,submission.shape[0]+1)
submission = submission[['Id', 'Solution']]
submission.to_csv('submission.csv',index=False)